In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Installing Libaries

In [ ]:
import os
folder_path = '/content/drive/MyDrive/MAJOR_PROJECT'
os.makedirs(folder_path, exist_ok=True)

In [ ]:
notebook_source = '/content/drive/MyDrive/MAJOR_PROJECT/L_Project/1_Preprocessing_Data.ipynb'
notebook_dest = os.path.join(folder_path, '1_Preprocessing_Data.ipynb')

!cp "{notebook_source}" "{notebook_dest}"
print(f"Notebook saved to: {notebook_dest}")

cp: cannot stat '/content/drive/MyDrive/MAJOR_PROJECT/01_Preprocessing_Data.ipynb': No such file or directory
Notebook saved to: /content/drive/MyDrive/MAJOR_PROJECT/01_Preprocessing_Data.ipynb


In [ ]:
! pip install chembl_webresource_client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 6.8 MB/s eta 0:00:00


## **Importing libraries**

In [ ]:
import pandas as pd
from chembl_webresource_client.new_client import new_client

## **Search for Target protein**

### Target search for GLP-1

In [ ]:
target = new_client.target
target_query = target.search('GLP-1')
targets = pd.DataFrame.from_dict(target_query)
targets

,cross_references,organism,pref_name,score,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Homo sapiens,Glucagon-like peptide 1 receptor,23.0,False,CHEMBL1784,"[{'accession': 'P43220', 'component_descriptio...",SINGLE PROTEIN,9606.0
1,[],Mus musculus,Glucagon-like peptide 1 receptor,23.0,False,CHEMBL1075290,"[{'accession': 'O35659', 'component_descriptio...",SINGLE PROTEIN,10090.0
2,[],Rattus norvegicus,Glucagon-like peptide 1 receptor,22.0,False,CHEMBL5862,"[{'accession': 'P32301', 'component_descriptio...",SINGLE PROTEIN,10116.0
3,[],Homo sapiens,Pro-glucagon,20.0,False,CHEMBL5736,"[{'accession': 'P01275', 'component_descriptio...",SINGLE PROTEIN,9606.0
4,[],Homo sapiens,Glucagon-like peptide 2 receptor,19.0,False,CHEMBL5844,"[{'accession': 'O95838', 'component_descriptio...",SINGLE PROTEIN,9606.0
...,...,...,...,...,...,...,...,...,...
4607,[],Homo sapiens,Baculoviral IAP repeat-containing protein 2/E3...,0.0,False,CHEMBL6066572,"[{'accession': 'P98170', 'component_descriptio...",PROTEIN FAMILY,9606.0
4608,[],Homo sapiens,Protein cereblon/Tubulin beta,0.0,False,CHEMBL6066847,"[{'accession': 'P68371', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,9606.0
4609,[],Mus musculus,RNA polymerase II,0.0,False,CHEMBL6066856,"[{'accession': 'P08775', 'component_descriptio...",PROTEIN COMPLEX,10090.0
4610,[],Severe acute respiratory syndrome coronavirus 2,Protein cereblon-SARS-Cov-2 polyprotein,0.0,False,CHEMBL6067603,"[{'accession': 'Q96SW2', 'component_descriptio...",PROTEIN-PROTEIN INTERACTION,2697049.0


In [ ]:
selected_target = targets.target_chembl_id[0]
selected_target

'CHEMBL1784'

In [ ]:
single_protein_targets = targets[targets['target_type'] == 'SINGLE PROTEIN']
display(single_protein_targets.head())

,cross_references,organism,pref_name,score,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Homo sapiens,Glucagon-like peptide 1 receptor,23.0,False,CHEMBL1784,"[{'accession': 'P43220', 'component_descriptio...",SINGLE PROTEIN,9606.0
1,[],Mus musculus,Glucagon-like peptide 1 receptor,23.0,False,CHEMBL1075290,"[{'accession': 'O35659', 'component_descriptio...",SINGLE PROTEIN,10090.0
2,[],Rattus norvegicus,Glucagon-like peptide 1 receptor,22.0,False,CHEMBL5862,"[{'accession': 'P32301', 'component_descriptio...",SINGLE PROTEIN,10116.0
3,[],Homo sapiens,Pro-glucagon,20.0,False,CHEMBL5736,"[{'accession': 'P01275', 'component_descriptio...",SINGLE PROTEIN,9606.0
4,[],Homo sapiens,Glucagon-like peptide 2 receptor,19.0,False,CHEMBL5844,"[{'accession': 'O95838', 'component_descriptio...",SINGLE PROTEIN,9606.0


Now I will count the number of activity datasets for each of these single protein targets. This might take some time depending on the number of targets.

In [ ]:
activity = new_client.activity
res = activity.filter(target_chembl_id=selected_target).filter(standard_type="IC50")

In [ ]:
df = pd.DataFrame.from_dict(res)

Save the resulting data to a CSV file bioactivity_data.csv

In [ ]:
df.shape

(349, 46)

In [ ]:
file_path = os.path.join(folder_path, 'L_DATASET_1/GLP_bioactivity_data_raw.csv')
df.to_csv(file_path, index=False)
print(f"CSV saved to: {file_path}")

CSV saved to: /content/drive/MyDrive/MAJOR_PROJECT/L_DATASET_1/GLP_bioactivity_data_raw.csv


### Handling Mising Data

Drop the missing value for the standard_value and canonical_smiles

In [ ]:
df2 = df[df.standard_value.notna()]
df2 = df2[df.canonical_smiles.notna()]
df2

/tmp/ipython-input-3852201246.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df2 = df2[df.canonical_smiles.notna()]


,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,None,1447512,[],CHEMBL874550,Inhibitory concentration required against huma...,B,None,None,BAO_0000190,...,Homo sapiens,Glucagon-like peptide 1 receptor,9606,None,None,IC50,nM,UO_0000065,None,14.0
1,None,None,1447515,[],CHEMBL829559,Inhibitory concentration against human GLP1 re...,B,None,None,BAO_0000190,...,Homo sapiens,Glucagon-like peptide 1 receptor,9606,None,None,IC50,nM,UO_0000065,None,0.14
2,None,None,1447518,[],CHEMBL874550,Inhibitory concentration required against huma...,B,None,None,BAO_0000190,...,Homo sapiens,Glucagon-like peptide 1 receptor,9606,None,None,IC50,nM,UO_0000065,None,5354.0
3,None,None,1447521,[],CHEMBL874550,Inhibitory concentration required against huma...,B,None,None,BAO_0000190,...,Homo sapiens,Glucagon-like peptide 1 receptor,9606,None,None,IC50,nM,UO_0000065,None,713.0
4,None,None,1447524,[],CHEMBL874550,Inhibitory concentration required against huma...,B,None,None,BAO_0000190,...,Homo sapiens,Glucagon-like peptide 1 receptor,9606,None,None,IC50,nM,UO_0000065,None,454.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
344,None,None,25751604,[],CHEMBL5442701,GPCR PRESTO-Tango dose-response in antagonist ...,F,None,None,BAO_0000190,...,Homo sapiens,Glucagon-like peptide 1 receptor,9606,None,None,IC50,uM,UO_0000065,None,10.0
345,None,None,25751642,[],CHEMBL5442701,GPCR PRESTO-Tango dose-response in antagonist ...,F,None,None,BAO_0000190,...,Homo sapiens,Glucagon-like peptide 1 receptor,9606,None,None,IC50,uM,UO_0000065,None,10.0
346,None,None,25751667,[],CHEMBL5442701,GPCR PRESTO-Tango dose-response in antagonist ...,F,None,None,BAO_0000190,...,Homo sapiens,Glucagon-like peptide 1 receptor,9606,None,None,IC50,uM,UO_0000065,None,10.0
347,"{'action_type': 'ANTAGONIST', 'description': '...",None,25751681,[],CHEMBL5442701,GPCR PRESTO-Tango dose-response in antagonist ...,F,None,None,BAO_0000190,...,Homo sapiens,Glucagon-like peptide 1 receptor,9606,None,None,IC50,uM,UO_0000065,None,3.02168


In [ ]:
len(df2.canonical_smiles.unique())

254

In [ ]:
df2_nr = df2.drop_duplicates(['canonical_smiles'])
df2_nr

,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,None,1447512,[],CHEMBL874550,Inhibitory concentration required against huma...,B,None,None,BAO_0000190,...,Homo sapiens,Glucagon-like peptide 1 receptor,9606,None,None,IC50,nM,UO_0000065,None,14.0
2,None,None,1447518,[],CHEMBL874550,Inhibitory concentration required against huma...,B,None,None,BAO_0000190,...,Homo sapiens,Glucagon-like peptide 1 receptor,9606,None,None,IC50,nM,UO_0000065,None,5354.0
3,None,None,1447521,[],CHEMBL874550,Inhibitory concentration required against huma...,B,None,None,BAO_0000190,...,Homo sapiens,Glucagon-like peptide 1 receptor,9606,None,None,IC50,nM,UO_0000065,None,713.0
4,None,None,1447524,[],CHEMBL874550,Inhibitory concentration required against huma...,B,None,None,BAO_0000190,...,Homo sapiens,Glucagon-like peptide 1 receptor,9606,None,None,IC50,nM,UO_0000065,None,454.0
5,None,None,1447527,[],CHEMBL874550,Inhibitory concentration required against huma...,B,None,None,BAO_0000190,...,Homo sapiens,Glucagon-like peptide 1 receptor,9606,None,None,IC50,nM,UO_0000065,None,804.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
344,None,None,25751604,[],CHEMBL5442701,GPCR PRESTO-Tango dose-response in antagonist ...,F,None,None,BAO_0000190,...,Homo sapiens,Glucagon-like peptide 1 receptor,9606,None,None,IC50,uM,UO_0000065,None,10.0
345,None,None,25751642,[],CHEMBL5442701,GPCR PRESTO-Tango dose-response in antagonist ...,F,None,None,BAO_0000190,...,Homo sapiens,Glucagon-like peptide 1 receptor,9606,None,None,IC50,uM,UO_0000065,None,10.0
346,None,None,25751667,[],CHEMBL5442701,GPCR PRESTO-Tango dose-response in antagonist ...,F,None,None,BAO_0000190,...,Homo sapiens,Glucagon-like peptide 1 receptor,9606,None,None,IC50,uM,UO_0000065,None,10.0
347,"{'action_type': 'ANTAGONIST', 'description': '...",None,25751681,[],CHEMBL5442701,GPCR PRESTO-Tango dose-response in antagonist ...,F,None,None,BAO_0000190,...,Homo sapiens,Glucagon-like peptide 1 receptor,9606,None,None,IC50,uM,UO_0000065,None,3.02168


## Data pre-processing

**Combine the 3 columns and bioactivity_class into a DataFrame**

In [ ]:
selection = ['molecule_chembl_id','canonical_smiles','standard_value']
df3 = df2_nr[selection]
df3

,molecule_chembl_id,canonical_smiles,standard_value
0,CHEMBL410972,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,14.0
2,CHEMBL265428,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,5354.0
3,CHEMBL439104,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,713.0
4,CHEMBL409873,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,454.0
5,CHEMBL410973,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,804.0
...,...,...,...
344,CHEMBL540612,COc1ccccc1N1CCN(CCCCNC(=O)c2ccc3ccccc3c2)CC1.Cl,10000.0
345,CHEMBL2164551,N#Cc1cc(F)cc(-c2nc(-c3ccc(F)cn3)no2)c1,10000.0
346,CHEMBL2018969,COc1cccc2c1c(NS(=O)(=O)c1ccc(Cl)s1)nn2Cc1cccc(...,10000.0
347,CHEMBL5075931,NCCc1cn(S(=O)(=O)c2c(Cl)nc3sccn23)c2ccccc12.O=...,3021.68


In [ ]:
df3.to_csv('GLP_bioactivity_data_preprocessed.csv', index=False)

In [ ]:
file_path = os.path.join(folder_path, 'L_DATASET_1/GLP_bioactivity_data_preprocessed.csv')
df3.to_csv(file_path, index=False)
print(f"CSV saved to: {file_path}")

CSV saved to: /content/drive/MyDrive/MAJOR_PROJECT/L_DATASET_1/GLP_bioactivity_data_preprocessed.csv


Labeling as active, inactive or intermediate

In [ ]:
df4 = pd.read_csv('/content/drive/MyDrive/MAJOR_PROJECT/L_DATASET_1/GLP_bioactivity_data_preprocessed.csv')

In [ ]:
bioactivity_threshold = []
for i in df4.standard_value:
  if float(i) >= 10000:
    bioactivity_threshold.append("inactive")
  elif float(i) <= 1000:
    bioactivity_threshold.append("active")
  else:
    bioactivity_threshold.append("intermediate")

In [ ]:
bioactivity_class = pd.Series(bioactivity_threshold, name='class')
df5 = pd.concat([df4, bioactivity_class], axis=1)
df5

,molecule_chembl_id,canonical_smiles,standard_value,class
0,CHEMBL410972,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,14.00,active
1,CHEMBL265428,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,5354.00,intermediate
2,CHEMBL439104,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,713.00,active
3,CHEMBL409873,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,454.00,active
4,CHEMBL410973,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,804.00,active
...,...,...,...,...
249,CHEMBL540612,COc1ccccc1N1CCN(CCCCNC(=O)c2ccc3ccccc3c2)CC1.Cl,10000.00,inactive
250,CHEMBL2164551,N#Cc1cc(F)cc(-c2nc(-c3ccc(F)cn3)no2)c1,10000.00,inactive
251,CHEMBL2018969,COc1cccc2c1c(NS(=O)(=O)c1ccc(Cl)s1)nn2Cc1cccc(...,10000.00,inactive
252,CHEMBL5075931,NCCc1cn(S(=O)(=O)c2c(Cl)nc3sccn23)c2ccccc12.O=...,3021.68,intermediate


In [ ]:
df5.head()

,molecule_chembl_id,canonical_smiles,standard_value,class
0,CHEMBL410972,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,14.0,active
1,CHEMBL265428,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,5354.0,intermediate
2,CHEMBL439104,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,713.0,active
3,CHEMBL409873,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,454.0,active
4,CHEMBL410973,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,804.0,active


In [ ]:
df5.to_csv('GLP_bioactivity_data_curated.csv', index=False)

In [ ]:
file_path = os.path.join(folder_path, 'L_DATASET_1/GLP_bioactivity_data_curated.csv')
df5.to_csv(file_path, index=False)
print(f"CSV saved to: {file_path}")

CSV saved to: /content/drive/MyDrive/MAJOR_PROJECT/L_DATASET_1/GLP_bioactivity_data_curated.csv
